# Packet aggregation using IP src and port dst

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

For the figure we will use only one day of the darknet IBR flows dataset. If you want to reproduce the results with all days, you can follow the same procedure.

To generate the input files, tshark and all necessary files are required. The following output is performed to obtain the desired fields:

```
for i in `ls -1 *.gz`
do
echo $i
zcat $i | tshark -r - -T fields -e frame.time_epoch -e ip.src -e tcp.dstport -E header=y -E separator=, -E quote=d -o ip.defragment:FALSE -E occurrence=f -Y "tcp" > ${i}-tcp.txt
zcat $i | tshark -r - -T fields -e frame.time_epoch -e ip.src -e udp.dstport -E header=y -E separator=, -E quote=d -o ip.defragment:FALSE -E occurrence=f -Y "udp" > ${i}-udp.txt
done
```

Then, all generated files must be concatenated into two files: one for TCP and another for UDP:

```sh
ls -art *tcp.txt | head -1 | xargs cat | head -1 > darknet-ibrflows-tcp.txt
ls -art *tcp.txt | xargs cat | grep -v ^frame.time_epoch >> darknet-ibrflows-tcp.txt
ls -art *udp.txt | head -1 | xargs cat | head -1 > darknet-ibrflows-udp.txt
ls -art *udp.txt | xargs cat | grep -v ^frame.time_epoch >> darknet-ibrflows-udp.txt
```

Files are now ready to be processed.

In [ ]:
csv_file_tcp ="darknet_20251010.pcap-tcp.txt"
csv_file_udp ="darknet_20251010.pcap-udp.txt"
output_file = "darknet_20251010-tcpyudp_ibrflows.pdf"

Load the CSV files as pandas DataFrames:

In [ ]:
df_tcp = pd.read_csv(csv_file_tcp, sep=',')

In [ ]:
df_udp = pd.read_csv(csv_file_udp, sep=',')

Change the title and labels of the plots as needed and the timestamp conversion.

In [ ]:
df_tcp.columns = ["timestamp", "ip_origen", "puerto_destino"]
df_tcp["timestamp"] = pd.to_numeric(df_tcp["timestamp"], errors="coerce")

In [ ]:
df_udp.columns = ["timestamp", "ip_origen", "puerto_destino"]
df_udp["timestamp"] = pd.to_numeric(df_udp["timestamp"], errors="coerce")

In [ ]:
simple_df_tcp = df_tcp.groupby(["ip_origen", "puerto_destino"]).size().reset_index(name="num_paquetes")
simple_df_udp = df_udp.groupby(["ip_origen", "puerto_destino"]).size().reset_index(name="num_paquetes")

Save the notebook to a files named `darknet-ibrflows-tcp.txt` and `darknet-ibrflows-udp.txt`.

In [ ]:
new_file_tcp = csv_file_tcp.replace('.txt', '') + '-ibrflow.csv'
new_file_udp = csv_file_udp.replace('.txt', '') + '-ibrflow.csv'

simple_df_tcp.to_csv(new_file_tcp, index=False)
simple_df_udp.to_csv(new_file_udp, index=False)

Plot the data

In [ ]:
key_1 = 'num_paquetes'

In [ ]:
# Function to process and draw a chart from a CSV file
def plot_packet_histogram(csv_file, ax, title, max_regular):
    # Read data
    df = pd.read_csv(csv_file, sep=',')

    bins = [0, 1, 5, 20, 100, 500, 1000, 5000, 10000, float('inf')]
    labels = ['1', '2-5', '6-20', '21-100', '101-500', '501-1000', '1001-5000', '5001-10000', '>10000']
    # labels = ['1', '2-5', '6-20', '21-100', '101-500', '501-1K', '1001-5K', '5001-10K', '>10K']

    # Classify each value into its interval for both columns
    df['bin_A_B'] = pd.cut(df[key_1], bins=bins, labels=labels, right=True, include_lowest=True)

    # Calculate totals per bin (summing packets)
    packets_A_B = df.groupby('bin_A_B', observed=False)[key_1].sum()

    # Create DataFrame and calculate percentages
    hist_df = pd.DataFrame({'Packets received': packets_A_B}).fillna(0).reindex(labels)
    total_packets = hist_df['Packets received'].sum()
    hist_df_percent = hist_df / total_packets * 100

    # Draw stacked chart on the provided axis
    hist_df_percent.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        width=0.85,
        color=['#1f77b4'],
        legend=False  # shared legend outside the function
    )

    # Add labels above bars
    for idx, entrada in enumerate(hist_df_percent['Packets received']):
        total = entrada
        label = f"{entrada:.1f}"
        ax.text(idx, total + 1, label, ha='center', va='bottom', fontsize=9, fontweight='bold', rotation=90)

    # Styling
    if max_regular:
        ax.set_ylim(0, 60)
    else:
        ax.set_ylim(0, 40)

    ax.set_title(title)
    # ax.set_xlabel('Packet in the flow (Received/Replied)')
    ax.set_xlabel('', fontsize=14)
    ax.set_ylabel('Total packets (%)', fontsize=12)

    # Rotation of xticks in x-axis
    ax.set_xticklabels(labels, rotation=90, fontsize=12)

In [ ]:
def generate_graphic(file1, title1, file2, title2, output_file, max_regular):
    fig, axes = plt.subplots(1, 2, figsize=(6, 6.5), sharey=True, constrained_layout=False)

    plot_packet_histogram(file1, axes[0], title1, max_regular)
    plot_packet_histogram(file2, axes[1], title2, max_regular)

    # Legend above, outside
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.995),
               ncol=1)

    # Shared X-axis label
    fig.supxlabel('Number of aggregated packets', fontsize=12, y=0.065)

    # Remove duplicated Y-axis label from right axis
    axes[1].set_ylabel('')

    # Compact margins and smaller horizontal spacing
    fig.subplots_adjust(left=0.13, right=0.98, top=0.89, bottom=0.28, wspace=0.06)

    fig.savefig(output_file, bbox_inches='tight', pad_inches=0.01)
    plt.show()

In [ ]:
generate_graphic(new_file_tcp, '(a) TCP packet aggregation', new_file_udp, '(b) UDP packet aggregation', output_file, max_regular=True)

Up to this point are the images used for the article; below are bar chart tests and other aggregations.

Chart of IBRflows that share the same source

In [ ]:
df_graph_bars = simple_df_tcp.groupby(["num_paquetes"]).size().reset_index(name='counts')

In [ ]:
# Sort the DataFrame by value (optional, affects only visual chart order)
df_graph_bars = df_graph_bars.sort_values(by='num_paquetes')

# Create bar chart
sns.barplot(data=df_graph_bars, x='num_paquetes', y='counts')

# Logarithmic scale on Y-axis
plt.yscale('log')

# Labels and title
plt.xlabel("Valor")
plt.ylabel("Cantidad")
plt.title("TCP IBRflows by number of aggregated packets")
plt.tight_layout()
plt.show()
